In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/Trang Ha/Documents/Data_Game/Cookie_Cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,9.0,False,False
1,337,gate_30,48.0,True,True
2,377,gate_40,2.0,False,False
3,483,gate_40,6.0,True,False
4,488,gate_40,NaN,True,True


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   userid          90189 non-null  int64  
 1   version         90189 non-null  str    
 2   sum_gamerounds  78741 non-null  float64
 3   retention_1     90189 non-null  bool   
 4   retention_7     90189 non-null  bool   
dtypes: bool(2), float64(1), int64(1), str(1)
memory usage: 2.2 MB


In [3]:
df.isnull().sum()

userid                0
version               0
sum_gamerounds    11448
retention_1           0
retention_7           0
dtype: int64

In [4]:
df = df.drop_duplicates(subset=['userid'], keep='first')

In [5]:
df["retention_1"].value_counts()

retention_1
False    49120
True     41069
Name: count, dtype: int64

In [6]:
df["retention_1"] = df["retention_1"].astype(int)
df["retention_7"] = df["retention_7"].astype(int)

In [7]:
df.describe()

,userid,sum_gamerounds,retention_1,retention_7
count,9.018900e+04,78741.000000,90189.000000,90189.000000
mean,4.998412e+06,54.318347,0.455366,0.247624
std,2.883286e+06,83.385139,0.498007,0.431635
min,1.160000e+02,0.000000,0.000000,0.000000
25%,2.512230e+06,5.000000,0.000000,0.000000
50%,4.995815e+06,18.000000,0.000000,0.000000
75%,7.496452e+06,69.000000,1.000000,0.000000
max,9.999861e+06,1069.000000,1.000000,1.000000


In [8]:
df["sum_gamerounds_new"] = df.groupby(
    ["version", "retention_1", "retention_7"]
)["sum_gamerounds"].transform(lambda group: group.fillna(group.median()))
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
0,116,gate_30,9.0,0,0,9.0
1,337,gate_30,48.0,1,1,48.0
2,377,gate_40,2.0,0,0,2.0
3,483,gate_40,6.0,1,0,6.0
4,488,gate_40,NaN,1,1,18.0


In [9]:
median_by_group = (
    df.groupby(["version", "retention_1", "retention_7"])["sum_gamerounds"]
      .transform("median")
)

mask_median = df["sum_gamerounds"].isna()

df.loc[mask_median, "sum_gamerounds_new"] = median_by_group[mask_median]

mask_zero = (
    df["sum_gamerounds"].isna() &
    (df["retention_1"] == False) &
    (df["retention_7"] == False)
)
df.loc[mask_zero, "sum_gamerounds_new"] = 0
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
0,116,gate_30,9.0,0,0,9.0
1,337,gate_30,48.0,1,1,48.0
2,377,gate_40,2.0,0,0,2.0
3,483,gate_40,6.0,1,0,6.0
4,488,gate_40,NaN,1,1,18.0


In [10]:
df[
    (df["version"] == "gate_40") &
    (df["retention_1"] == 1) &
    (df["retention_7"] == 1) &
    (df["sum_gamerounds"].isna())
]

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
4,488,gate_40,NaN,1,1,18.0
466,48201,gate_40,NaN,1,1,18.0
533,55250,gate_40,NaN,1,1,18.0
863,96745,gate_40,NaN,1,1,18.0
1021,113087,gate_40,NaN,1,1,18.0
...,...,...,...,...,...,...
89588,9934927,gate_40,NaN,1,1,18.0
89732,9951192,gate_40,NaN,1,1,18.0
89767,9954781,gate_40,NaN,1,1,18.0
89768,9954937,gate_40,NaN,1,1,18.0


In [11]:
df.drop(columns=["sum_gamerounds"], inplace=True)


In [13]:
df.rename(columns={"sum_gamerounds_new": "sum_gamerounds"}, inplace=True)
df.head()

,userid,version,retention_1,retention_7,sum_gamerounds
0,116,gate_30,0,0,9.0
1,337,gate_30,1,1,48.0
2,377,gate_40,0,0,2.0
3,483,gate_40,1,0,6.0
4,488,gate_40,1,1,18.0


In [14]:
summary = (
    df.groupby("version")
      .agg(
          Players=("userid","count"),
          D1_Retention=("retention_1","mean"),
          D7_Retention=("retention_7","mean")
      )
)

summary

,Players,D1_Retention,D7_Retention
version,,,
gate_30,44700,0.458367,0.250626
gate_40,45489,0.452417,0.244675


In [15]:
lift_d1 = (
    summary.loc["gate_40", "D1_Retention"]
    - summary.loc["gate_30", "D1_Retention"]
)

lift_d7 = (
    summary.loc["gate_40", "D7_Retention"]
    - summary.loc["gate_30", "D7_Retention"]
)

print(lift_d1)
print(lift_d7)

-0.005949822517753001
-0.005951861509110312


In [16]:

from statsmodels.stats.proportion import proportions_ztest
# Hàm thực hiện Two-Proportion Z-Test
def retention_ztest(df, retention_col):
    
    # Số người retained của mỗi version
    count = [
        df[df["version"] == "gate_30"][retention_col].sum(),
        df[df["version"] == "gate_40"][retention_col].sum()
    ]

    # Tổng số người chơi của mỗi version
    nobs = [
        len(df[df["version"] == "gate_30"]),
        len(df[df["version"] == "gate_40"])
    ]

    # Two-Proportion Z-Test
    stat, pvalue = proportions_ztest(count, nobs)

    # Kết luận
    decision = "Significant" if pvalue < 0.05 else "Not Significant"

    return stat, pvalue, decision

# D1 Retention
stat_d1, pvalue_d1, decision_d1 = retention_ztest(df, "retention_1")

# D7 Retention
stat_d7, pvalue_d7, decision_d7 = retention_ztest(df, "retention_7")

In [17]:
experiment_summary = pd.DataFrame({
    "Metric": ["Day 1 Retention", "Day 7 Retention"],
    "Z-statistic": [stat_d1, stat_d7],
    "p-value": [pvalue_d1, pvalue_d7],
    "Decision": [decision_d1, decision_d7]
})

experiment_summary

,Metric,Z-statistic,p-value,Decision
0,Day 1 Retention,1.793914,0.072827,Not Significant
1,Day 7 Retention,2.070470,0.038408,Significant


In [18]:
experiment_summary.to_csv("C:/Users/Trang Ha/Documents/Data_Game/experiment_summary.csv",index=False)